# 04 - Final Multi-Seed Training

Retrains all models using frozen best hyperparameters from HPO
across all assets and horizons with multiple random seeds.

- Seeds: [123, 456, 789]
- Total: 9 models × 2 horizons × 12 assets × 3 seeds = 648 runs

Results are aggregated as mean ± std per model-asset-horizon combination.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

from src.utils.config import ProjectConfig
from src.experiments.final_runner import (
    run_final_single,
    run_final_for_model_asset,
    run_all_final,
)
from src.models import list_models

config = ProjectConfig('..')
print(f'Models: {list_models()}')
print(f'Seeds: {config.get_eval_seeds()}')
print(f'Horizons: {config.get_horizons()}')

total = (len(list_models()) * len(config.get_horizons()) * 
         sum(len(config.get_assets_for_category(c)) for c in config.get_categories()) *
         len(config.get_eval_seeds()))
print(f'\nTotal training runs: {total}')

## Run Final Training

In [ ]:
# # Option 1: Run a single model-asset-horizon-seed
# metrics = run_final_single(config, 'LSTM', 'crypto', 'BTCUSDT', 
#                            'BTCUSDT_H1.csv', horizon=4, seed=123)
# print(metrics)

# # Option 2: Run all seeds for one model-asset-horizon
# # agg = run_final_for_model_asset(config, 'LSTM', 'crypto', 'BTCUSDT',
# #                                 'BTCUSDT_H1.csv', horizon=4)
# # print(agg)

# # Option 3: Run all final training (uncomment to execute)
run_all_final(config)

# # Option 4: Run for specific models only
# run_all_final(config, models_filter=['N-HiTS', 'DLinear','TimesNet'])

print('To run final training, uncomment one of the options above.')
print('Or run from CLI: python src/main.py final')

## Inspect Results

In [ ]:
import json
import pandas as pd
from pathlib import Path
from src.evaluation.aggregate import load_aggregated_metrics, format_metric_string

final_dir = config.get_path('results_dir') / 'final'

if final_dir.exists():
    agg_files = list(final_dir.rglob('aggregated.json'))
    print(f'Found {len(agg_files)} aggregated result files\n')
    
    rows = []
    for agg_file in agg_files:
        parts = agg_file.relative_to(final_dir).parts
        model, category, asset, horizon = parts[0], parts[1], parts[2], parts[3]
        
        agg = load_aggregated_metrics(agg_file)
        row = {'model': model, 'category': category, 'asset': asset, 'horizon': horizon}
        
        for metric_name, metric_data in agg.items():
            if isinstance(metric_data, dict) and 'mean' in metric_data:
                row[metric_name] = format_metric_string(
                    metric_data['mean'], metric_data['std']
                )
        rows.append(row)
    
    if rows:
        results_df = pd.DataFrame(rows)
        display(results_df)
else:
    print('No final training results yet. Run final training first.')

print('\nProceed to 05_model_comparison.ipynb for benchmarking.')